# Question 11 - 274. H-Index

Given an array of integers `citations` where `citations[i]` is the number of citations a researcher received for their `i-th` paper, return the researcher's **h-index**.

According to the definition of h-index on Wikipedia: The h-index is defined as the maximum value of `h` such that the given researcher has published at least `h` papers that have each been cited at least `h` times.

**Example 1:**

    Input: citations = [3,0,6,1,5]
    Output: 3
    Explanation: [3,0,6,1,5] means the researcher has 5 papers in total and each of them had received 3, 0, 6, 1, 5 citations respectively.
    Since the researcher has 3 papers with at least 3 citations each and the remaining two with no more than 3 citations each, their h-index is 3.

**Example 2:**

    Input: citations = [1,3,1]
    Output: 1

**Constraints:**

- `n == citations.length`
- `1 <= n <= 5000`
- `0 <= citations[i] <= 1000`

---
## 🎯 Interview Walkthrough: Brute Force → Optimized

**Pattern:** Counting / buckets (cap every value at n)

📘 **Revise first:** [Pattern E · counting](./0-concepts-array-and-string.ipynb) · [Part 3.9 · sorting](./0-concepts-array-and-string.ipynb) in the concepts notebook

| Approach | Time | Space |
|---|---|---|
| 1️⃣ Brute force (test every h) | O(n²) | O(1) |
| 1½ Sort, then scan | O(n log n) | O(1)–O(n) |
| 2️⃣ Optimized (count buckets) | O(n) | O(n) |

> 🧠 = plain-English intuition (how you'd explain it to a friend) · 🗣️ = **interview script**: what to say out loud at each step.

### 🧠 The problem in plain English

A researcher's **h-index** is a single number that summarises "how many good papers do they have?". The definition:

> **h** is the **largest** number such that the researcher has **at least h papers**, each cited **at least h times**.

`citations = [3, 0, 6, 1, 5]`
- h = 3? Papers with ≥ 3 citations: **3, 6, 5** → 3 papers ✅
- h = 4? Papers with ≥ 4 citations: **6, 5** → only 2 papers ❌
- So **h = 3**.

**Everyday picture: a podium of staircases.** Sort the papers from most to least cited and line them up. Walk along the line: the 1st paper needs ≥ 1 citation, the 2nd needs ≥ 2, the 3rd needs ≥ 3… Keep going while each paper clears its bar. **The number of papers that cleared their bar is h.**

```
rank:        1   2   3   4   5
citations:   6   5   3   1   0
needs ≥:     1   2   3   4   5
cleared?     ✅  ✅  ✅  ❌      → h = 3
```

**One fact unlocks the fastest solution:** h can never exceed the number of papers, **n**. You can't have "at least 10 papers" if you only wrote 5.

### Step 0 · Clarify before coding

🗣️ *"I want the largest h such that at least h papers have at least h citations each. h is between 0 and n, the number of papers."*

| Question to ask | Answer | Why it matters |
|---|---|---|
| Range of h? | `0 ≤ h ≤ n`. | Lets us use an array of size n+1 instead of sorting. |
| All zeros? | h = 0. | |
| May I modify / sort the input? | Ask! | If not, sort a copy or use a separate count array. |
| Already sorted? | No. (If it were, that's LC 275, solved with binary search.) | |

### Step 1 · Brute force: the "obvious" way

**Plain-English idea:** test each candidate h from the biggest possible (n) downward. For each, **count** how many papers have ≥ h citations. The first h that passes is the answer.

🗣️ *"The definition translates directly: for each candidate h from n down to 0, count papers with at least h citations; the first h where the count is at least h is the answer."*

**Why is it slow?** Up to n+1 candidates, and each count scans all n papers → **n × n**. With n = 5000 that's 25 million steps. It works, but we're recounting the same papers every time.

In [1]:
class SolutionBrute:
    def hIndex(self, citations: list[int]) -> int:
        n = len(citations)
        for h in range(n, -1, -1):                          # biggest candidate first
            if sum(1 for c in citations if c >= h) >= h:    # full scan per candidate
                return h
        return 0

#### Step 1½ · Sort first (the "podium" from the intro)

Sort **descending**. The paper at position `i` (0-based) is the `(i+1)`-th most cited. If it has at least `i+1` citations, then the top `i+1` papers all do, so h is at least `i+1`. Keep going until that fails.

🗣️ *"Sorting puts the papers in exactly the order the definition cares about, so the answer becomes a simple scan. That's O(n log n) because of the sort. Can I avoid sorting?"*

In [2]:
class SolutionSort:
    def hIndex(self, citations: list[int]) -> int:
        cs = sorted(citations, reverse=True)      # most cited first
        h = 0
        while h < len(cs) and cs[h] >= h + 1:     # paper #(h+1) clears the bar h+1
            h += 1
        return h

### Step 2 · Optimize: the "aha" moment

We don't actually need the full order. For each h, we only need **how many papers have ≥ h citations**.

> 💡 **Aha:** since h can't be more than n, a paper with 1,000 citations counts **exactly the same** as a paper with n citations: both clear every possible bar. So **cap every citation count at n**, and use **buckets** numbered 0 to n.

1. **Bucket count:** `count[c]` = how many papers have exactly `c` citations (with anything above n put in bucket n).
2. **Sweep from the top** (h = n, n−1, …): keep a running total `papers` = number of papers with ≥ h citations (bucket h plus everything above it). The **first** h where `papers ≥ h` is the answer.

This is a **counting sort** idea: when values live in a small known range, counting beats comparing.

🗣️ *"Since h is at most n, I can cap citations at n and count them into n+1 buckets. Then I sweep h from n downward, accumulating how many papers have at least h citations. The first h where that running total reaches h is the answer. That's O(n) time and O(n) space for the buckets."*

#### 🤔 Why this approach, and why not the others?

| Option | Verdict | Plain-English reason |
|---|---|---|
| Test every h by rescanning | ❌ | O(n²). Recounts everything for each candidate. |
| **Sort + scan** | ✅ safest first answer | O(n log n), short, and O(1) extra memory if sorting in place is allowed. |
| Binary search on h | ⚠️ | "≥ h papers with ≥ h" gets harder as h grows, so it works, but each check is O(n) → O(n log n). No gain over sorting. |
| Dict of counts | ⚠️ | Works, but the values are small integers in a known range, so an array is simpler and faster. |
| **Count buckets, capped at n** | ✅ **best time** | O(n). Uses the "h ≤ n" bound. Costs O(n) memory. |
| Heap | ❌ | No benefit; popping up to n items costs O(n log n). |

**The trade-off to say out loud:** the bucket solution spends **O(n) memory to save time**. If memory is tight, sorting in place is O(n log n) time with O(1) extra memory.

In [3]:
class Solution:
    def hIndex(self, citations: list[int]) -> int:
        n = len(citations)
        count = [0] * (n + 1)               # buckets 0..n
        for c in citations:
            count[min(c, n)] += 1           # anything >= n behaves like n

        papers = 0                          # papers with >= h citations
        for h in range(n, -1, -1):
            papers += count[h]
            if papers >= h:
                return h
        return 0

### Step 3 · Toy example walkthrough (tell it like a story)

`citations = [3, 0, 6, 1, 5]`, n = 5

**Bucket counting (cap at 5):** 3 → bucket 3, 0 → bucket 0, 6 → **bucket 5** (capped), 1 → bucket 1, 5 → bucket 5

| bucket h | 0 | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|---|
| count[h] | 1 | 1 | 0 | 1 | 0 | 2 |

**Sweep from the top:**

| h | count[h] | papers with ≥ h | papers ≥ h? |
|---|---|---|---|
| 5 | 2 | 2 | 2 ≥ 5 ❌ |
| 4 | 0 | 2 | 2 ≥ 4 ❌ |
| 3 | 1 | **3** | 3 ≥ 3 ✅ → **h = 3** |

**Narrated:** "Is the h-index 5? Only 2 papers have 5 or more citations, so no. 4? Still only 2 papers. 3? Adding the paper with exactly 3 citations gives 3 papers with at least 3, so yes. The h-index is 3."

In [4]:
cases = [
    ([3, 0, 6, 1, 5], 3),
    ([1, 3, 1], 1),
    ([0], 0),
    ([0, 0, 0], 0),
    ([100], 1),                 # capping matters here
    ([1000, 1000, 1000], 3),
    ([4, 4, 0, 0], 2),
]
for cits, expected in cases:
    for Impl in (SolutionBrute, SolutionSort, Solution):
        assert Impl().hIndex(cits) == expected, Impl.__name__
    print(cits, "->", expected)

import random
for _ in range(500):
    cits = [random.randint(0, 12) for _ in range(random.randint(1, 10))]
    assert Solution().hIndex(cits) == SolutionBrute().hIndex(cits), cits
print("All tests passed ✅")

[3, 0, 6, 1, 5] -> 3
[1, 3, 1] -> 1
[0] -> 0
[0, 0, 0] -> 0
[100] -> 1
[1000, 1000, 1000] -> 3
[4, 4, 0, 0] -> 2
All tests passed ✅


### Step 4 · Complexity in plain words, and wrap up

| | Time | Extra memory | In plain words |
|---|---|---|---|
| Test every h | O(n²) | O(1) | n+1 candidates × a full count each. |
| Sort + scan | O(n log n) | O(1) in place / O(n) copy | The sort dominates. |
| **Count buckets** | **O(n)** | **O(n)** | One pass to fill buckets, one pass to sweep. |

**Why O(n) time?** Filling the buckets touches each paper once. The sweep visits at most n+1 buckets. Each step is constant work.

**Why O(n) memory?** The bucket array has n+1 slots.

**Edge cases to mention:** all zeros (0), one paper with many citations (1, which is where capping matters), every paper hugely cited (n).

**Likely follow-ups:**
- *Citations already sorted (LC 275)?* → Binary search for the first index `i` with `citations[i] ≥ n − i`. O(log n).
- *Papers arriving one at a time?* → Keep the buckets and the current h. Each new paper can raise h by at most 1.

---

#### 🗣️ Full interview script (about 60 seconds)

*"The h-index is the largest h such that h papers each have at least h citations, so h is between 0 and n.*

*Brute force tries every h and counts qualifying papers, which is O(n²). Sorting descending turns it into a single scan: the i-th paper must have at least i+1 citations. That's O(n log n).*

*To get O(n), I use the fact that h can't exceed n. A paper with more than n citations is equivalent to one with exactly n, so I cap values at n and count them into n+1 buckets. Then I sweep h from n downward with a running total of papers having at least h citations, and return the first h where that total reaches h.*

*That's O(n) time and O(n) extra space. If memory mattered more, I'd use the in-place sort version."*